# R1. Blind baseline (15 min)  |  Client brief: **History**
> *"I have transcribed snippets from three newspapers of a fictional port town, Aldermere, 1880s. I need each one coded for how it frames the dockworkers' strike. The archive grant review is Friday."*

**Your job:** write a zero-shot prompt in **5 minutes**, run it, and see where you land on the board.

**How the arena works**
- Your prompt can use placeholders: `{text}` (the item), `{labels}`, `{codebook}` (the official definitions), `{schema}` (extraction).
- The harness adds one output rule so everyone is parsed the same way. You do **not** need to ask for a specific format.
- Every run costs real money from your allowance and appears on the board. **Write your hypothesis in `note=` before you run.**
- The hidden test set stays with the instructor. Scores you see here are on the *dev* set.

In [1]:
# ===== Run this cell first =====
import os, sys, hashlib, json
KIT_URL = "https://github.com/atsizelti/prompt_engineering.git"   # public repo with the kit
KIT_DIR = "arena_kit/prompt_arena_kit"                            # folder inside the clone
import importlib.util
if os.path.exists("arena_kit"):
    os.system("git -C arena_kit pull -q")                # always refresh a copy cloned earlier in this session
elif importlib.util.find_spec("arena") is None:
    os.system(f"git clone -q {KIT_URL} arena_kit")
if os.path.exists(KIT_DIR) and KIT_DIR not in sys.path:
    sys.path.insert(0, KIT_DIR)
import arena

NICK      = "denomeno"      # pick a nickname (no real name needed)
PROVIDER  = "openai"             # "openai", or "mock" (practice with no key)
ENDPOINT  = "https://script.google.com/macros/s/AKfycbxdVYynFuPHaHg73kTFzL5R0TYXrPA8Tr0adMEsb0LKUS5vpDgysktWLPTTcLWrS7Jz/exec"     # the class board (already filled in: you do not need to change it)
TOKEN     = "f-BEqiAfHhAc2O-b"
ALLOWANCE = 0.03                # dollars you may spend (the whole class shares one small budget; re-running this cell does not reset it)

if os.environ.get("ARENA_MOCK"):
    PROVIDER = "mock"
if os.environ.get("ARENA_NO_BOARD"):                  # used by the instructor's offline tests
    ENDPOINT = TOKEN = ""
KEY = None
if PROVIDER != "mock":
    try:
        from google.colab import userdata
        KEY = userdata.get("OPENAI_API_KEY")
    except Exception:
        KEY = os.environ.get("OPENAI_API_KEY")
try:                                                  # the board address and password, set once as Colab secrets
    from google.colab import userdata
    ENDPOINT = ENDPOINT or userdata.get("BOARD_ENDPOINT")
    TOKEN = TOKEN or userdata.get("BOARD_TOKEN")
except Exception:
    ENDPOINT = ENDPOINT or os.environ.get("BOARD_ENDPOINT", "")
    TOKEN = TOKEN or os.environ.get("BOARD_TOKEN", "")
ENDPOINT, TOKEN = "".join(ENDPOINT.split()), "".join(TOKEN.split())   # remove hidden spaces and line breaks from pasting
arena.setup(NICK, provider=PROVIDER, api_key=KEY, endpoint=ENDPOINT or None,
            token=TOKEN or None, allowance_usd=ALLOWANCE)
arena.set_round("R1")

[arena] ready: student=denomeno provider=openai model=gpt-5-nano allowance=$0.03 (already spent $0.000) logging=on
[arena] models you may use: gpt-5-nano, gpt-4o-mini, gpt-5.4-nano, gpt-5.4-mini


In [16]:
arena.peek("A", k=4)        # the brief, the labels, and 4 sample snippets (labels hidden)

Task A: The Harbor Gazette: strike framing  [History]
A historian transcribed snippets from three newspapers of a fictional port town (1880s). Code how each snippet frames the dockworkers' strike.
Labels: SYMPATHETIC, HOSTILE, NEUTRAL_REPORT, MERCHANT_COST, OFF_TOPIC
(50 items in the 'dev' set; showing 4)

[H022]  (The Harbor Gazette) The ferry timetable to Southwick was altered after the tide turned early, and several excursionists grumbled in the pleasantest possible manner.

[H002]  (The Free Tide) Aldermere's merchants talk solemnly of losses, but none of them has yet explained how a man is to feed six children on the same fourteen shillings fixed in the year of hard winter and harder bread.

[A035]  (The Aldermere Courier) The Whitby steamer Anne Bellamy was diverted to Harrowgate on Tuesday.

[H008]  (The Aldermere Courier) It is a curious sort of fairness that leaves ships waiting, merchants counting losses, and the public pestered by loafers who call themselves injured men; the

### Write your prompt
Keep it **minimal first**. Add instructions only when the errors show you need them.

In [17]:
PROMPT = """Classify this newspaper snippet by how it frames the dockworkers' strike, the labels are: SYMPATHETIC, HOSTILE, NEUTRAL_REPORT, MERCHANT_COST, OFF_TOPIC
Options: {labels}

Text: {text}"""

HYPOTHESIS = "H: a longer prompt will get the easy cases and irony."   # write yours before you run
res = arena.evaluate(PROMPT, task="A", condition="blind", note=HYPOTHESIS)
res.summary()

[arena] STOPPED: allowance used up. This run was not scored or logged.


BudgetExceeded: Allowance of $0.03 used up.

In [4]:
# Look at what went wrong. Patterns, not single items.
for e in res.errors(6):
    print(f"[{e['id']}] gold={e['gold']}  predicted={e['pred']}\n    {e['text'][:160]}\n")

[H005] gold=HOSTILE  predicted=MERCHANT_COST
    Such tenderness as the strike deserves may be measured by the damage it has done to merchants, carters, and the very wives they pretend to defend. To hear them 

[H011] gold=NEUTRAL_REPORT  predicted=OFF_TOPIC
    The town clerk entered into the minutes that two lighters and a schooner were awaiting discharge at Eastern Wharf, a detail included in the ordinary shipping li

[H012] gold=NEUTRAL_REPORT  predicted=OFF_TOPIC
    The Harbour Road tally clerk entered six carts of Baltic timber against the account of Wetherell & Prowse, while the tide held too low for unloading at the east

[A032] gold=NEUTRAL_REPORT  predicted=OFF_TOPIC
    The steam-crane arrived by barge on Monday and stands, as yet unused, on the quay.

[A033] gold=NEUTRAL_REPORT  predicted=OFF_TOPIC
    Mr. Penhallow, the builder, attended the Council to describe the crane's capacity, which he gave as twelve tons.

[A035] gold=NEUTRAL_REPORT  predicted=OFF_TOPIC
    The Whi

### Before the reveal: predict
Where did you land among the class (top / middle / bottom)? Which of your choices made the biggest difference?
Write it down. The next round tests whether you were right.

In [ ]:
MY_GUESS = "I think my prompt is in the ____ of the class because ____"
print(MY_GUESS)